# LAB 10 — K-Means Clustering
## Finding Customer Segments, Choosing K and Avoiding the Traps on Real Data

## Student Information
- **Name: Nainsy Chawla**
- **Student ID: 023-24-0261**
- **Section: H**
- **GitHub:  https://github.com/Nainsy-chawla/ML_Lab_Practise**
- **Kaggle: https://www.kaggle.com/nainsychawla**
- **Dataset:** Mall Customers (Kaggle)
- **Dataset Source:** https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python

### Before You Start
- This lab does not reteach K-Means. It makes you use and test what you saw in the lecture. Each task gives the steps, not the code; the **Key definitions** table in each Part lists what you need.
- There is **no target column**: no train/test split and no accuracy. Judge every result by distances, silhouette, plots and common sense.
- Use `SEED = 42` everywhere. Write 1–3 lines of interpretation under every table and plot. A plot without an explanation earns no marks.
- Place `Mall_Customers.csv` in the same folder as this notebook (or upload it to Colab).
- Time: about 2 hours 15 minutes of work. Use the remaining time to finish answers, run the notebook top to bottom and submit.

---

## Part 0 — Setup and First Look (10 minutes)

Load the data and look at it before you cluster anything.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Unsupervised learning | Learning from data that has no labels. The goal is to find structure (groups), not to predict a column. | `No y, no train_test_split.` |
| Cluster | A group of points that are close to each other and far from the other groups. | `model.labels_ gives the cluster number of every row.` |
| Feature | One column used to describe each row (here Age, Income, Spending). | `df[['Income','Spending']].values` |

**First cell:** import everything you need (NumPy, Pandas, Matplotlib, KMeans, StandardScaler, silhouette_score, silhouette_samples, adjusted_rand_score) and set `SEED = 42`.

In [33]:
# YOUR CODE HERE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import adjusted_rand_score

# Set random seed
SEED = 42

### Task 0.1 — Load and check the data

Load Mall_Customers.csv and understand it in one cell.

1. Load the CSV. Rename 'Annual Income (k$)' to Income and 'Spending Score (1-100)' to Spending (shorter names).
2. Print the shape, the number of missing values, the number of duplicate rows and the count of each Gender.
3. Show describe() for Age, Income and Spending.

In [ ]:
# YOUR CODE HERE
df = pd.read_csv('Mall_Customers.csv')

df.rename(columns={
    'Annual Income (k$)': 'Income',
    'Spending Score (1-100)': 'Spending'
}, inplace=True)

print("First five rows:")
display(df.head())

print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nGender Count:")
print(df['Gender'].value_counts())

print("\nStatistical Summary:")
display(df[['Age', 'Income', 'Spending']].describe())

### Task 0.2 — Look at the data before clustering

Use your eyes first.

1. Draw one scatter plot of Spending vs Income, with axis labels and a title.

In [ ]:
# YOUR CODE HERE
# YOUR CODE HERE

plt.figure(figsize=(8, 5))

plt.scatter(df['Income'], df['Spending'])

plt.title('Spending vs Income')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score')

plt.tight_layout()
plt.show()

**Questions** (answer with your own numbers)

(a) How many groups can you see by eye? Describe them in one line each (e.g. 'high income, low spending').

(b) Why can we not use accuracy or a train/test split to judge a clustering result?

**Your answers:**

(a) 
1.Low income, high spending – Customers with low income but high spending scores.

2.Low income, low spending – Customers with low income and low spending scores.

3.Medium income, medium spending – A dense group of customers with moderate income and spending scores.

4.High income, high spending – Customers with high income and high spending scores.

5.High income, low spending – Customers with high income but low spending scores

(b) 
Clustering is an unsupervised learning technique, so there are no predefined correct labels. Accuracy requires actual labels for comparison, while train/test splitting is generally used in supervised learning. Therefore, we use measures such as Inertia and Silhouette Score to evaluate clustering.

---

## Part 1 — K-Means From Scratch (30 minutes)

In the lecture you did K-Means by hand. Here you turn those steps into code and check your own function against scikit-learn. Using `KMeans()` inside your own function is not allowed.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| K | The number of clusters. You choose it before running. | `KMeans(n_clusters=5)` |
| Centroid | The centre (mean) of a cluster. It is a point, not always a real row. | `model.cluster_centers_` |
| Distance | 1 feature: \|x - m\|. 2 or more features: Euclidean, sqrt of the sum of squared differences. | `np.abs(a-b)  or  np.sqrt(((a-b)**2).sum(axis=1))` |
| Assign step | Give each point to its nearest centroid. | `d.argmin(axis=1)` |
| Update step | Move each centroid to the mean of the points assigned to it. | `X[labels==k].mean(axis=0)` |
| Convergence | Stop when no point changes cluster (centroids stop moving). | `np.allclose(new_c, c)` |
| Inertia | Sum of squared distances from every point to its own centroid. Smaller = tighter clusters. | `model.inertia_` |

### Task 1.1 — Lecture Example 1 in code (one feature)

Reproduce the 9-number example from the lecture (2, 4, 10, 12, 3, 20, 30, 11, 25) with K = 2 and starting centroids 4 and 11.

1. Write a loop that, in every iteration, computes the distances, assigns the clusters, updates the centroids and prints both clusters and the new centroids.
2. Stop when no centroid changes. Print the final centroids and the inertia.
3. Compare with the lecture slides (final centroids 7 and 25, inertia 150).

**Hints:**
- Distance matrix: `np.abs(x[:, None] - m[None, :])` has shape (9, 2).
- Cluster of each point: `dist.argmin(axis=1)`.

In [ ]:
# YOUR CODE HERE

x = np.array([2, 4, 10, 12, 3, 20, 30, 11, 25])
m = np.array([4.0, 11.0])

iteration = 0

while True:
    iteration += 1

    dist = np.abs(x[:, None] - m[None, :])
    labels = dist.argmin(axis=1)

    new_m = np.array([
        x[labels == k].mean() for k in range(2)
    ])

    print("Iteration:", iteration)
    print("Cluster C1:", x[labels == 0])
    print("Cluster C2:", x[labels == 1])
    print("New centroids:", new_m)
    print("------------------------")

    if np.allclose(new_m, m):
        m = new_m
        break

    m = new_m

# Calculate final inertia
final_dist = np.abs(x - m[labels]) ** 2
inertia = final_dist.sum()

print("Final centroids:", m)
print("Final inertia:", inertia)


### Task 1.2 — Your own K-Means function vs scikit-learn

Turn your loop into a function, then show it gives the same answer as scikit-learn.

1. Write `my_kmeans(X, init_centroids, max_iter=100)` that works for any number of features and returns labels, centroids, inertia and the number of iterations. Test it on Example 1: it must give centroids 7 and 25 and inertia 150.
2. Use X2 = Income and Spending (200 x 2) with the first 5 rows as starting centroids. Run `my_kmeans` and `KMeans(n_clusters=5, init=<the same 5 rows>, n_init=1, tol=0, algorithm='lloyd')`, then compare the inertia and the centroids.

In [ ]:
# YOUR CODE HERE


def my_kmeans(X, init_centroids, max_iter=100):

    X = np.array(X, dtype=float)
    centroids = np.array(init_centroids, dtype=float)

    for iteration in range(1, max_iter + 1):

        # Calculate Euclidean distances
        distances = np.sqrt(
            ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
        )

        # Assign points to nearest centroids
        labels = distances.argmin(axis=1)

        # Update centroids
        new_centroids = centroids.copy()

        for k in range(len(centroids)):
            if np.any(labels == k):
                new_centroids[k] = X[labels == k].mean(axis=0)

        # Check convergence
        if np.allclose(new_centroids, centroids):
            centroids = new_centroids
            break

        centroids = new_centroids

    # Calculate inertia
    distances = np.sqrt(
        ((X - centroids[labels]) ** 2).sum(axis=1)
    )
    inertia = (distances ** 2).sum()

    return labels, centroids, inertia, iteration


In [ ]:
# YOUR CODE HERE

x_example = np.array([2, 4, 10, 12, 3, 20, 30, 11, 25]).reshape(-1, 1)
init_example = np.array([[4], [11]])

labels_ex, centers_ex, inertia_ex, iterations_ex = my_kmeans(
    x_example, init_example
)

print("Example 1 Results")
print("Final Centroids:", centers_ex.flatten())
print("Inertia:", inertia_ex)
print("Iterations:", iterations_ex)


# Test on Mall Customers dataset

X2 = df[['Income', 'Spending']].values

# First five rows as initial centroids
init = X2[:5]

# Run our own K-Means
my_labels, my_centers, my_inertia, my_iterations = my_kmeans(X2, init)

# Run scikit-learn K-Means
sk_model = KMeans(
    n_clusters=5,
    init=init,
    n_init=1,
    tol=0,
    algorithm='lloyd'
)

sk_labels = sk_model.fit_predict(X2)

# Compare results
print("\nMall Customers Comparison")

print("My K-Means Inertia:", my_inertia)
print("Scikit-learn Inertia:", sk_model.inertia_)

print("\nMy K-Means Iterations:", my_iterations)
print("Scikit-learn Iterations:", sk_model.n_iter_)

print("\nMy K-Means Centroids:\n", my_centers)
print("\nScikit-learn Centroids:\n", sk_model.cluster_centers_)

print("\nLabels Match:", np.array_equal(my_labels, sk_labels))
print("Centroids Match:", np.allclose(my_centers, sk_model.cluster_centers_))
print("Inertia Match:", np.isclose(my_inertia, sk_model.inertia_))


**Questions** (answer with your own numbers)

(a) Example 1: the point 10 started in cluster C2 but moved to C1 in iteration 2. Explain why, using the centroid values.

(b) What makes the loop stop? Why is the stopping guaranteed even though the final answer is not always the best one?

(c) Did your function match scikit-learn? If anything differed, explain why.

**Your answers:**

(a) After the first iteration, the updated centroids became 3 and 18. Now, point 10 was closer to centroid 3 because its distances were 7 and 8, respectively. Therefore, it moved to C1.

(b) The loop stops when the centroids no longer change or when the maximum number of iterations is reached. K-Means usually converges to a stable solution, but it may find a local minimum instead of the global best solution. The result can depend on the initial centroids.

(c) Yes, my K-Means function matched Scikit-learn exactly. Both produced the same cluster labels, centroids, inertia of 44448.4554, and completed in 4 iterations. This happened because both algorithms used the same initial centroids and Lloyd's K-Means algorithm.

---

## Part 2 — Choosing K: Elbow and Silhouette (25 minutes)

Nobody tells us K. Use the two lecture tools, the elbow and the silhouette, then calculate one silhouette by hand to prove you understand it.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Elbow method | Plot inertia for K = 2, 3, 4 ... Inertia always falls as K grows, so look for the K where the fall suddenly becomes small (the bend). | `model.inertia_ in a loop over K` |
| a | Average distance from a point to the other points in its own cluster (smaller = better). | `d[same_cluster].mean()` |
| b | Average distance from the point to the points of the nearest other cluster (larger = better). | `min over other clusters` |
| Silhouette s | s = (b - a) / max(a, b). Close to +1: well placed. Near 0: on the border. Negative: probably in the wrong cluster. | `silhouette_samples(X, labels)` |
| Average silhouette | The mean of s over all points. The K with the highest value is a good candidate. | `silhouette_score(X, labels)` |

### Task 2.1 — Elbow and silhouette for K = 2 to 10

Decide how many customer groups exist using Income and Spending (X2).

1. For K = 2 to 10 fit `KMeans(n_clusters=K, n_init=10, random_state=SEED)` and record the inertia and the average silhouette.
2. Show a table (K, Inertia, Silhouette) and two plots: elbow curve and silhouette bars.
3. Print the K with the highest silhouette.

In [ ]:

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd
import matplotlib.pyplot as plt

results = []

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=SEED
    )

    labels = model.fit_predict(X2)

    inertia = model.inertia_
    silhouette = silhouette_score(X2, labels)

    results.append({
        'K': k,
        'Inertia': inertia,
        'Silhouette': silhouette
    })

results_df = pd.DataFrame(results)

print("K-Means Results:")
display(results_df.round(4))

# Find K with highest silhouette
best_k = results_df.loc[
    results_df['Silhouette'].idxmax(), 'K'
]

print("K with highest silhouette:", best_k)

# Elbow curve
plt.figure(figsize=(8, 5))
plt.plot(results_df['K'], results_df['Inertia'], marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(range(2, 11))
plt.grid(True)
plt.show()

# Silhouette scores
plt.figure(figsize=(8, 5))
plt.bar(results_df['K'], results_df['Silhouette'])
plt.title('Average Silhouette Score for Different K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Average Silhouette Score')
plt.xticks(range(2, 11))
plt.grid(axis='y')
plt.show()


### Task 2.2 — Silhouette by hand

Prove that you understand the formula by calculating it yourself.

1. Fit K = 5. Write a function that returns a, b and s for one point, using only NumPy distances. Apply it to customer 1 (row 0) and compare with `silhouette_samples`.

In [ ]:

from sklearn.metrics import silhouette_samples

# Fit K-Means with K = 5
model = KMeans(n_clusters=5, n_init=10, random_state=SEED)
labels = model.fit_predict(X2)

def calculate_silhouette(X, labels, point_index):
    point = X[point_index]
    own_cluster = labels[point_index]

    # Calculate distances from this point to all points
    distances = np.sqrt(((X - point) ** 2).sum(axis=1))

    # Calculate a: average distance within the same cluster
    same_cluster = (labels == own_cluster)
    same_cluster[point_index] = False

    a = distances[same_cluster].mean()

    # Calculate b: minimum average distance to another cluster
    other_clusters = set(labels) - {own_cluster}
    other_distances = []

    for cluster in other_clusters:
        cluster_points = distances[labels == cluster]
        other_distances.append(cluster_points.mean())

    b = min(other_distances)

    # Calculate silhouette
    s = (b - a) / max(a, b)

    return a, b, s


# Customer 1 is at row 0
a, b, s = calculate_silhouette(X2, labels, 0)

# Compare with sklearn
sk_scores = silhouette_samples(X2, labels)
sk_s = sk_scores[0]

print("Customer 1 — Silhouette by Hand")
print("a (same cluster distance):", a)
print("b (nearest other cluster distance):", b)
print("Silhouette by hand:", s)

print("\nScikit-learn Silhouette:", sk_s)
print("Values Match:", np.isclose(s, sk_s))


**Questions** (answer with your own numbers)

(a) Which K does the elbow suggest and which K does the silhouette suggest? Do they agree? Quote your numbers.

(b) What is the silhouette of customer 1, and what does it tell you about where that customer sits?

(c) Why should we never choose K only by the lowest inertia?

**Your answers:**

(a) The elbow method suggests K = 5 because the reduction in inertia becomes smaller after K = 5. The silhouette method also suggests K = 5, with the highest score of 0.5539. Therefore, both methods agree.

(b) Customer 1 has a silhouette score of 0.4170. This positive score indicates that the customer is reasonably placed in its own cluster and is closer to its own cluster than to the nearest other cluster.

(c) Inertia always decreases when the number of clusters increases. Therefore, choosing K only by the lowest inertia could lead to too many clusters. We should also consider the elbow method and silhouette score to find a suitable number of clusters.

---

## Part 3 — Starting Points and Stability (15 minutes)

The lecture said 'starting points matter — run several starts and keep the best'. Here you measure how much.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Local minimum | A solution that cannot be improved by the assign/update loop but is not the best possible one. | `A higher final inertia than the best run.` |
| init='random' | Pick K random rows as the starting centroids. | `KMeans(init='random')` |
| init='k-means++' | Pick starting centroids that are far from each other. This is the default and is usually better. | `KMeans(init='k-means++')` |
| n_init | How many different starts to try. scikit-learn keeps the run with the lowest inertia. | `KMeans(n_init=10)` |

### Task 3.1 — 30 different seeds

Measure how often K-Means finds the best answer.

1. With K = 5 on X2, run KMeans 30 times (random_state = 0 to 29) in two settings: `init='random'` with `n_init=1`, and `init='k-means++'` with `n_init=10`.
2. For each setting print the lowest and the highest final inertia and the number of distinct values (round to 1 decimal).

In [ ]:

random_inertias = []
kmeanspp_inertias = []

for seed in range(30):

    # Random initialization
    random_model = KMeans(
        n_clusters=5,
        init='random',
        n_init=1,
        random_state=seed
    )
    random_model.fit(X2)
    random_inertias.append(random_model.inertia_)

    # K-Means++ initialization
    kmeanspp_model = KMeans(
        n_clusters=5,
        init='k-means++',
        n_init=10,
        random_state=seed
    )
    kmeanspp_model.fit(X2)
    kmeanspp_inertias.append(kmeanspp_model.inertia_)


# Print results
print("Random Initialization (n_init=1)")
print("Lowest Inertia:", min(random_inertias))
print("Highest Inertia:", max(random_inertias))
print("Distinct Inertia Values:",
      len(set(round(x, 1) for x in random_inertias)))

print("\nK-Means++ Initialization (n_init=10)")
print("Lowest Inertia:", min(kmeanspp_inertias))
print("Highest Inertia:", max(kmeanspp_inertias))
print("Distinct Inertia Values:",
      len(set(round(x, 1) for x in kmeanspp_inertias)))


**Questions** (answer with your own numbers)

(a) How many different final inertia values did each setting produce?

(b) Why can the same data and the same K give different answers?

(c) How do k-means++ and n_init solve this problem? Use your numbers.

**Your answers:**

(a) Random initialization with n_init=1 produced 14 distinct inertia values, while K-Means++ with n_init=10 produced only 1 distinct value.
 
(b) K-Means starts with initial centroids. Different starting points can produce different cluster assignments and lead to different local minima. Therefore, the same dataset and K can produce different final inertia values.

(c) K-Means++ selects well-separated initial centroids, increasing the chances of finding a good solution. The n_init parameter runs K-Means multiple times and keeps the result with the lowest inertia. In our experiment, random initialization produced 14 different values, ranging from 44448.46 to 91761.98, whereas K-Means++ with n_init=10 consistently produced 44448.46.

---

## Part 4 — Scaling: Units Decide the Clusters (20 minutes)

The lecture warned: 'Rs 5000 would swamp 3 visits'. You will prove it by changing only the unit of Income.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Feature scaling | Putting all features on a similar scale so that no feature wins the distance only because its numbers are bigger. | `StandardScaler().fit_transform(X)` |
| StandardScaler | z = (x - mean) / std. Every column gets mean 0 and standard deviation 1. | `StandardScaler()` |
| Adjusted Rand Index (ARI) | Agreement between two groupings. 1 = same groups, about 0 = no better than random. It ignores the cluster numbers (cluster 2 in one run can be cluster 4 in another). | `adjusted_rand_score(a, b)` |

### Task 4.1 — Change the unit, change the clusters

Use Age and Income with K = 4. Income is in thousands of dollars (15 to 137). Make a second copy where Income is in dollars (multiply by 1000).

1. Cluster four versions: raw k$, raw dollars, scaled k$, scaled dollars (StandardScaler, `n_init=10`, `random_state=SEED`).
2. Print the ARI between raw k$ and raw dollars, and between scaled k$ and scaled dollars.
3. Draw Age vs Income (in k$) twice: coloured by the raw-dollar clusters and by the scaled clusters.

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

# 1. Age and Income in thousands (k$)
X_age_income_k = df[['Age', 'Income']].values

# 2. Age and Income in dollars
X_age_income_dollars = X_age_income_k.copy()
X_age_income_dollars[:, 1] = X_age_income_dollars[:, 1] * 1000

# 3. Scale the k$ data
X_scaled_k = StandardScaler().fit_transform(X_age_income_k)

# 4. Scale the dollars data
X_scaled_dollars = StandardScaler().fit_transform(X_age_income_dollars)


# Apply K-Means to all four versions
model_raw_k = KMeans(n_clusters=4, n_init=10, random_state=SEED)
labels_raw_k = model_raw_k.fit_predict(X_age_income_k)

model_raw_dollars = KMeans(n_clusters=4, n_init=10, random_state=SEED)
labels_raw_dollars = model_raw_dollars.fit_predict(X_age_income_dollars)

model_scaled_k = KMeans(n_clusters=4, n_init=10, random_state=SEED)
labels_scaled_k = model_scaled_k.fit_predict(X_scaled_k)

model_scaled_dollars = KMeans(n_clusters=4, n_init=10, random_state=SEED)
labels_scaled_dollars = model_scaled_dollars.fit_predict(X_scaled_dollars)


# Compare cluster assignments using ARI
print("ARI: Raw k$ vs Raw Dollars:",
      adjusted_rand_score(labels_raw_k, labels_raw_dollars))

print("ARI: Raw k$ vs Scaled Dollars:",
      adjusted_rand_score(labels_raw_k, labels_scaled_dollars))

print("ARI: Scaled k$ vs Scaled Dollars:",
      adjusted_rand_score(labels_scaled_k, labels_scaled_dollars))


In [ ]:

plt.figure(figsize=(8, 5))
plt.scatter(
    df['Age'],
    df['Income'],
    c=labels_raw_dollars,
    cmap='viridis'
)
plt.title('Age vs Income — Raw Dollar Clusters')
plt.xlabel('Age')
plt.ylabel('Annual Income (k$)')
plt.show()


plt.figure(figsize=(8, 5))
plt.scatter(
    df['Age'],
    df['Income'],
    c=labels_scaled_dollars,
    cmap='viridis'
)
plt.title('Age vs Income — Scaled Clusters')
plt.xlabel('Age')
plt.ylabel('Annual Income (k$)')
plt.show()


**Questions** (answer with your own numbers)

(a) What happened when only the unit of Income changed? Explain it with the distance formula: which feature controls the distance now? What did scaling fix?

(b) In a real project with no labels, how will you decide whether to scale? Write your rule in one or two sentences.

**Your answers:**

(a) When Income was changed from thousands to dollars, its values became 1000 times larger. Since K-Means uses Euclidean distance,
Income dominated the distance because its numerical values became much larger than Age. This changed the cluster assignments, giving an ARI of 0.4897 between the two raw datasets. StandardScaler fixed this by bringing both features to a similar scale. The scaled datasets produced identical clusters, with an ARI of 1.0.

(b) I will check the ranges and units of the numerical features before applying K-Means. If features have different scales or units, I will use StandardScaler and compare clustering quality using silhouette scores and domain knowledge.

---

## Part 5 — Outliers (15 minutes)

The lecture said 'one extreme point drags a centroid'. You will add one extreme customer, see the damage, and catch it with a simple distance rule.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Outlier | A point far away from all the others. K-Means uses means and squared distances, so one outlier has a big effect. | `A very large distance to the nearest centroid.` |
| Distance to own centroid | How far a point is from the centre of its cluster. Large = the point does not fit any group well. | `model.transform(X).min(axis=1)` |
| mean + 3 std rule | Flag a point if its distance is larger than the mean distance plus 3 standard deviations. | `d > d.mean() + 3*d.std()` |

### Task 5.1 — One extreme customer

Add one extra row to X2: Income = 400, Spending = 99. Use K = 5 on Income and Spending.

1. Fit KMeans (`n_init=10`, `random_state=SEED`) without and with the extra customer.
2. Print the cluster sizes and the centroids (sorted by income) for both fits.
3. Draw the two clusterings side by side.

In [ ]:

# Add one extreme customer: Income = 400, Spending = 99
extra_customer = np.array([[400, 99]])

X2_with_outlier = np.vstack([X2, extra_customer])

# K-Means without the extra customer
model_original = KMeans(
    n_clusters=5,
    n_init=10,
    random_state=SEED
)
labels_original = model_original.fit_predict(X2)

# K-Means with the extra customer
model_outlier = KMeans(
    n_clusters=5,
    n_init=10,
    random_state=SEED
)
labels_outlier = model_outlier.fit_predict(X2_with_outlier)


# Function to sort centroids by Income
def sorted_centroids(model):
    return model.cluster_centers_[
        np.argsort(model.cluster_centers_[:, 0])
    ]


print("Original Cluster Sizes:")
print(np.bincount(labels_original))

print("\nOriginal Centroids (Income, Spending):")
print(sorted_centroids(model_original))

print("\nCluster Sizes With Outlier:")
print(np.bincount(labels_outlier))

print("\nCentroids With Outlier (Income, Spending):")
print(sorted_centroids(model_outlier))


### Task 5.2 — Catch it with a distance rule

Use the model fitted on the original 200 customers.

1. Compute every customer's distance to its nearest centroid, and the distance of the extra customer to the nearest centroid of the same model.
2. Apply the rule mean + 3 std (mean and std from the 200 real customers). Is the extra customer flagged? Are any real customers flagged?

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original clustering
axes[0].scatter(
    X2[:, 0], X2[:, 1],
    c=labels_original,
    cmap='viridis'
)
axes[0].scatter(
    model_original.cluster_centers_[:, 0],
    model_original.cluster_centers_[:, 1],
    marker='X',
    s=200,
    color='red',
    label='Centroids'
)
axes[0].set_title('Original Customers')
axes[0].set_xlabel('Annual Income (k$)')
axes[0].set_ylabel('Spending Score')
axes[0].legend()

# Clustering with outlier
axes[1].scatter(
    X2_with_outlier[:-1, 0],
    X2_with_outlier[:-1, 1],
    c=labels_outlier[:-1],
    cmap='viridis'
)
axes[1].scatter(
    X2_with_outlier[-1, 0],
    X2_with_outlier[-1, 1],
    color='black',
    marker='*',
    s=250,
    label='Extra Customer'
)
axes[1].scatter(
    model_outlier.cluster_centers_[:, 0],
    model_outlier.cluster_centers_[:, 1],
    marker='X',
    s=200,
    color='red',
    label='Centroids'
)
axes[1].set_title('With One Extreme Customer')
axes[1].set_xlabel('Annual Income (k$)')
axes[1].set_ylabel('Spending Score')
axes[1].legend()

plt.tight_layout()
plt.show()

# Distance of each original customer to nearest centroid
real_distances = model_original.transform(X2).min(axis=1)

# Distance of extra customer to nearest original centroid
outlier_distance = model_original.transform(extra_customer).min()

# Calculate threshold using original customers only
mean_distance = real_distances.mean()
std_distance = real_distances.std()

threshold = mean_distance + 3 * std_distance

# Find flagged original customers
flagged_real = real_distances > threshold

# Check extra customer
outlier_flagged = outlier_distance > threshold

print("Mean Distance:", mean_distance)
print("Standard Deviation:", std_distance)
print("Threshold (Mean + 3 Std):", threshold)

print("\nExtra Customer Distance:", outlier_distance)
print("Is Extra Customer Flagged?", outlier_flagged)

print("\nNumber of Real Customers Flagged:", flagged_real.sum())
print("Flagged Real Customer Indices:", np.where(flagged_real)[0])



**Questions** (answer with your own numbers)

(a) What did the single extreme customer do to the clusters? Use the sizes and centroids.

(b) Did the rule catch the extra customer? If real customers were flagged too, are they errors or simply unusual customers?

(c) Name two ways to deal with outliers and one risk of simply deleting them.

**Your answers:**

(a) Adding the extreme customer (400, 99) changed the cluster sizes from [81, 39, 22, 35, 23] to [100, 1, 23, 39, 38]. It formed a separate cluster with its own centroid (400, 99). The other centroids also changed, showing that one extreme customer can affect the K-Means clustering.

(b) Yes, the rule caught the extra customer. Its distance was 313.9153, which was greater than the threshold of 34.8024. Four real customers (indices 196, 197, 198, and 199) were also flagged. They may simply be unusual customers and should be investigated before being considered errors.

(c) Two ways to deal with outliers are:
1. Use a method such as DBSCAN that can identify noise points.
2. Apply a transformation or use a robust scaling method to reduce the effect of extreme values.

---

## Part 6 — Name the Segments and Wrap-Up (20 minutes)

Turn your clusters into segments that a marketing team could use.

**Key definitions**

| Term | What it means | How you write / read it |
|---|---|---|
| Segment profile | The average Age, Income and Spending of each cluster. It tells you who is in the group. | `df.groupby('Cluster')[['Age','Income','Spending']].mean()` |
| Deliverable | A file someone else can use without running your notebook. | `df.to_csv('customer_segments.csv', index=False)` |

### Task 6.1 — Profile and name the segments

Use the K you chose in Part 2 and the Income and Spending features.

1. Fit the final model (`n_init=10`, `random_state=SEED`) and add a Cluster column to the DataFrame.
2. Show the size and the mean Age, Income and Spending of each cluster.
3. Give each cluster a short name (e.g. 'High income, low spending') and one marketing action. Store the name in a Segment column.

In [ ]:

# Final K-Means model
final_model = KMeans(
    n_clusters=5,
    n_init=10,
    random_state=SEED
)

final_labels = final_model.fit_predict(X2)

# Add cluster labels to the original DataFrame
df['Cluster'] = final_labels

# Calculate cluster sizes
cluster_sizes = df['Cluster'].value_counts().sort_index()

# Calculate average Age, Income and Spending
cluster_profile = df.groupby('Cluster')[
    ['Age', 'Income', 'Spending']
].mean()

# Combine sizes and profiles
cluster_summary = cluster_profile.copy()
cluster_summary.insert(0, 'Size', cluster_sizes)

print("Customer Segment Profiles:")
display(cluster_summary.round(2))

# Define segment names
segment_names = {
    0: 'Average Income, Average Spending',
    1: 'High Income, High Spending',
    2: 'Low Income, High Spending',
    3: 'High Income, Low Spending',
    4: 'Low Income, Low Spending'
}

# Add Segment column
df['Segment'] = df['Cluster'].map(segment_names)

# Display customer data with segment names
display(df[['CustomerID', 'Age', 'Income', 'Spending',
            'Cluster', 'Segment']].head())

# Display segment counts
print("\nSegment Counts:")
print(df['Segment'].value_counts())



**Segment names and actions** (one line per cluster)

| Cluster | Segment name | Marketing action |
|---|---|---|
| 0 | | |
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |

### Task 6.2 — Save and summarise

Produce a file someone else can use without running your notebook.

1. Save `customer_segments.csv` with CustomerID, Age, Gender, Income, Spending, Cluster and Segment.
2. Write a 4–6 line Executive Summary (Markdown): the K you chose and the evidence for it, the segments and their sizes, how you handled scaling and outliers, and one limitation.

In [ ]:

# Select required columns
export_df = df[
    ['CustomerID', 'Age', 'Gender', 'Income',
     'Spending', 'Cluster', 'Segment']
]

# Save CSV file
export_df.to_csv('customer_segments.csv', index=False)

print("Customer segments saved successfully!")
print("Total records:", len(export_df))


**Executive Summary** (4–6 lines)

*K-Means clustering was applied to the Mall Customers dataset using Income and Spending, and five clusters were selected. The choice was supported by the elbow method and the highest silhouette score of 0.5539 at K = 5. The five segments were named based on their average income and spending, with a total of 200 customers. Feature scaling was used to prevent differences in measurement units from dominating distances, and an extreme customer was detected using the mean + 3 standard deviations distance rule. One limitation is that K-Means is sensitive to outliers and initial centroids, and the resulting segments may not represent actual customer behaviour perfectly.*

---

### Before you submit
- [ ] Restart the kernel and **Run All**: the notebook runs top to bottom without errors.
- [ ] Every table and plot has 1–3 lines of interpretation below it.
- [ ] All questions are answered with your own numbers.
- [ ] Student Information block is filled in.
- [ ] Notebook and `customer_segments.csv` are pushed to your GitHub repository.